In [66]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3
import requests

In [104]:
gemini_baseurl = "https://generativelanguage.googleapis.com/v1beta/openai/"
load_dotenv(override=True)
api_key = os.getenv("GOOGLE_API_KEY")
client = OpenAI(base_url=gemini_baseurl, api_key=api_key)

In [10]:
#Tạo db
DB = 'Nhietdo.db'
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute(
                'CREATE TABLE IF NOT EXISTS weather (city TEXT PRIMARY KEY, temperature INTEGER)'
            )
    conn.commit()

### Sử Dụng SQLite3 hoặc Api 

In [16]:
# weather_data = [
#         ('hanoi', 20),
#         ('hochiminh', 25),
#         ('haiphong', 21),
#         ('da nang', 22),
#         ('can tho', 24),
#         ('quang ninh', 19),
#         ('nha trang', 23),
#         ('phu quoc', 26),
#         ('vung tau', 24),
#         ('hue', 21)
#     ]
# with sqlite3.connect(DB) as conn:
#     cursor = conn.cursor()
#     cursor.executemany(
#         'INSERT OR REPLACE INTO weather (city, temperature) VALUES (?, ?)',
#         weather_data
#     )
#     conn.commit()

In [ ]:

# def get_weather_infor(destination_city):
#     print(f"DATABASE TOOL CALLED: Getting weather of {destination_city}", flush=True)
#     city = destination_city.strip().lower().replace(" ", "")

#     with sqlite3.connect(DB) as conn:
#         # tạo trỏ
#         cursor = conn.cursor()
        
#         # kiểm tra nếu đã có dữ liệu
#         cursor.execute('SELECT temperature FROM weather WHERE city = ?', (city.lower(),))
#         result = cursor.fetchone() # Kiểm tra nếu có kết quả
#         return f"Weather in {city} is {result[0]}°C" if result else "No weather data available for this city"

In [105]:
def get_weather_infor(destination_city):
    print(f"API CALL: Getting real-time weather of {destination_city}", flush=True)
    search_query = f"{destination_city.strip()}"
    #API Key từ OpenWeatherMap
    API_KEY = os.getenv("OPENWEATHER_API_KEY") 
    base_url = "http://api.openweathermap.org/data/2.5/weather"
    
    # Tham số gọi API
    params = {
        "q": search_query,
        "appid": API_KEY,
        "units": "metric", 
        "lang": "vi" 
    }
    
    try:
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if response.status_code == 200:
            temp = data['main']['temp']
            desc = data['weather'][0]['description']
            return f"Thời tiết tại {destination_city} hiện là {temp}°C, {desc}."
        else:
            return f"Không tìm thấy thông tin thời tiết cho {destination_city}."
    except Exception as e:
        return f"Lỗi khi kết nối API: {str(e)}"

In [112]:
get_weather_infor("HaiPhong")

API CALL: Getting real-time weather of HaiPhong


'Thời tiết tại HaiPhong hiện là 26.95°C, mây đen u ám.'

In [95]:
import requests

# Thay bằng key Default của bạn
TEST_API_KEY = "546a9688b86cde8f1716886e4fed055c" 
city = "Hanoi"
url = f"http://api.openweathermap.org/data/2.5/weather?q={city}&appid={TEST_API_KEY}&units=metric&lang=vi"

response = requests.get(url)
print("Status Code:", response.status_code)
print("Kết quả:", response.json())

Status Code: 200
Kết quả: {'coord': {'lon': 105.8412, 'lat': 21.0245}, 'weather': [{'id': 803, 'main': 'Clouds', 'description': 'mây cụm', 'icon': '04n'}], 'base': 'stations', 'main': {'temp': 27, 'feels_like': 30.12, 'temp_min': 27, 'temp_max': 27, 'pressure': 1006, 'humidity': 84, 'sea_level': 1006, 'grnd_level': 1005}, 'visibility': 10000, 'wind': {'speed': 6.45, 'deg': 149, 'gust': 13.07}, 'clouds': {'all': 67}, 'dt': 1776269808, 'sys': {'type': 1, 'id': 9308, 'country': 'VN', 'sunrise': 1776206279, 'sunset': 1776251708}, 'timezone': 25200, 'id': 1581130, 'name': 'Hà Nội', 'cod': 200}


In [73]:
tools = [{"type": "function",
           "function": {
               "name": "get_weather_infor",
               "description": "LUÔN sử dụng hàm này để lấy thông tin thời tiết. Hàm hiểu các tên tỉnh thành tiếng Việt: Hà Nội, TP. Hồ Chí Minh, Hải Phòng, Đà Nẵng, Cần Thơ, Quảng Ninh, Nha Trang, Phú Quốc, Vũng Tàu, Huế",
               "parameters": {
                   "type": "object",
                   "properties": {
                       "destination_city": {
                           "type": "string",
                           "description": "Tên tỉnh thành Việt Nam bằng tiếng Việt hoặc tiếng Anh. Ví dụ: 'Hà Nội', 'Ha Noi', 'Hồ Chí Minh', 'Ho Chi Minh', 'Đà Nẵng', 'Da Nang', v.v.",
                       },
                   },
                   "required": ["destination_city"],
               }
           }
           }]
tools


[{'type': 'function',
  'function': {'name': 'get_weather_infor',
   'description': 'LUÔN sử dụng hàm này để lấy thông tin thời tiết. Hàm hiểu các tên tỉnh thành tiếng Việt: Hà Nội, TP. Hồ Chí Minh, Hải Phòng, Đà Nẵng, Cần Thơ, Quảng Ninh, Nha Trang, Phú Quốc, Vũng Tàu, Huế',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': "Tên tỉnh thành Việt Nam bằng tiếng Việt hoặc tiếng Anh. Ví dụ: 'Hà Nội', 'Ha Noi', 'Hồ Chí Minh', 'Ho Chi Minh', 'Đà Nẵng', 'Da Nang', v.v."}},
    'required': ['destination_city']}}}]

In [74]:
def handle_tool_call(message):
    cities = []
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_weather_infor':
            argurments = json.loads(tool_call.function.arguments)
            city = argurments.get('destination_city')
            cities.append(city)
            temp_details = get_weather_infor(city)
            responses.append({
                'role': 'tool',
                'content': temp_details,
                'tool_call_id': tool_call.id
            })
    return responses, cities


In [75]:
system_msg = '''Bạn là trợ lý hữu ích.
Hãy trả lời ngắn gọn, lịch sự, không quá 1 câu.
Luôn luôn chính xác. Nếu bạn không biết câu trả lời, hãy nói thẳng ra.

**IMPORTANT**: Khi user hỏi về thời tiết bất kỳ tỉnh thành nào (bằng tiếng Việt hoặc tiếng Anh), 
bạn PHẢI gọi hàm get_weather_infor với tên tỉnh thành đó.

Các tỉnh thành có dữ liệu thời tiết:
- Hà Nội = Hanoi = ha noi
- TP. Hồ Chí Minh = Ho Chi Minh = ho chi minh = Sài Gòn
- Hải Phòng = Haiphong = hai phong
- Đà Nẵng = Da Nang = da nang
- Cần Thơ = Can Tho = can tho
- Quảng Ninh = Quang Ninh = quang ninh
- Nha Trang = nha trang
- Phú Quốc = Phu Quoc = phu quoc
- Vũng Tàu = Vung Tau = vung tau
- Huế = Hue = hue

Ví dụ:
- User: "Thời tiết ở Hà Nội?" → Gọi get_weather_infor("Hà Nội")
- User: "Weather in Da Nang?" → Gọi get_weather_infor("Da Nang")'''


In [77]:
# def Client(message, history):
#     history = [
#         {'role': h['role'], 'content': h['content']} for h in history]
#     messages = [
#         {'role': 'system', 'content': system_msg},
#         {'role': 'user', 'content': message}
#     ]
#     response = client.chat.completions.create(
#         model = 'gemini-2.5-flash-lite',
#         messages= messages,
#         tools=tools
#     )
    
#     if response.choices[0].finish_reason=="tool_calls":
#         message = response.choices[0].message
#         responses, cities = handle_tool_call(message)
#         messages.append(message)
#         messages.extend(responses)
#         response = client.chat.completions.create(model='gemini-2.5-flash-lite', messages=messages)
#     return response.choices[0].message.content


In [27]:
groq_api_key = os.getenv('GROQ_API_KEY')
groq_url = "https://api.groq.com/openai/v1"
client = OpenAI(base_url=groq_url, api_key=groq_api_key)

In [78]:
def Client(message, history):
    history = [
        {'role': h['role'], 'content': h['content']} for h in history]
    messages = [
        {'role': 'system', 'content': system_msg}, 
        *history,
        {'role': 'user', 'content': message}
    ]
    response = client.chat.completions.create(
        model = 'openai/gpt-oss-120b',
        messages= messages,
        tools=tools
    )
    
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_call(message)
        messages.append(message)
        messages.extend(responses)
        response = client.chat.completions.create(model='openai/gpt-oss-120b', messages=messages, tools=tools)
    return response.choices[0].message.content


In [79]:
gr.ChatInterface(
    fn=Client,
    title="Chatbot and weatherVN func"
).launch()


* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


API CALL: Getting real-time weather of Hà Nội


In [64]:
# Model Configuration
models_config = {
    'Gemini 2.5 Flash Lite': {
        'base_url': "https://generativelanguage.googleapis.com/v1beta/openai/",
        'api_key_env': 'GOOGLE_API_KEY',
        'model_name': 'gemini-2.5-flash-lite',
        'max_iterations': 1  # Gemini: single tool call
    },
    'Groq GPT-OSS 120B': {
        'base_url': "https://api.groq.com/openai/v1",
        'api_key_env': 'GROQ_API_KEY',
        'model_name': 'openai/gpt-oss-120b',
        'max_iterations': 5  # Groq: multiple tool calls
    }
}

def Client_with_model(message, history, model_name='Gemini 2.5 Flash Lite'):
    """Client function với model selector"""
    config = models_config.get(model_name)
    if not config:
        return "Model không hợp lệ. Vui lòng chọn model khác."
    
    # Setup client theo config
    api_key = os.getenv(config['api_key_env'])
    temp_client = OpenAI(base_url=config['base_url'], api_key=api_key)
    model_id = config['model_name']
    max_iterations = config['max_iterations']
    
    # Prepare messages
    history = [{'role': h['role'], 'content': h['content']} for h in history]
    messages = [
        {'role': 'system', 'content': system_msg}, 
        *history,
        {'role': 'user', 'content': message}
    ]
    
    # First API call
    response = temp_client.chat.completions.create(
        model=model_id,
        messages=messages,
        tools=tools
    )
    
    # Handle tool calls (loop for Groq, once for Gemini)
    iteration = 0
    while response.choices[0].finish_reason == "tool_calls" and iteration < max_iterations:
        message_obj = response.choices[0].message
        responses, cities = handle_tool_call(message_obj)
        messages.append(message_obj)
        messages.extend(responses)
        
        response = temp_client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=tools
        )
        iteration += 1
    
    return response.choices[0].message.content


In [65]:
gr.ChatInterface(
    fn=Client_with_model,
    additional_inputs=[
        gr.Dropdown(
            choices=list(models_config.keys()),
            value='Gemini 2.5 Flash Lite',
            label='Chọn Model'
        )
    ],
    title="FlightAI - Weather Assistant"
).launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.
